# A robot that asks people questions: order effects, uncertainty and trust

All data in this notebook are **simulated** from illustrative populations; no human-subject data are used.

1. Do people's answers depend on the order of the robot's questions? (QQ test, model comparison)
2. A human-model ensemble gives the robot a predicted answer *and* its uncertainty.
3. `ask_or_act` turns that into a decision: ask the person, or act.
4. Trust as a qubit: how successes and failures move a person's trust, on the Bloch sphere.

In [ ]:
# Use the installed package, or the source tree when running from a clone
try:
    import qlcog
except ImportError:
    import sys, pathlib; sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src')); import qlcog
print('qlcog', qlcog.__version__)

## 1. Question order

In [ ]:
import numpy as np
from qlcog.applications.robotics import HRI_DOMAINS, domain_models
from qlcog.families.order_effects import qq_test

domain = 'object_clarification'
print('A:', HRI_DOMAINS[domain]['A']); print('B:', HRI_DOMAINS[domain]['B'])
people = domain_models(domain)['QL']                       # simulated quantum-like population
rng = np.random.default_rng(0)
counts = people.sample(None, 300, rng)                     # 300 people per question order
print(counts)
print('QQ test (q, z, p):', qq_test(counts))

## 2. Human-model ensemble

In [ ]:
from qlcog.applications.robotics import HumanModelEnsemble
from qlcog.families.order_effects import QuantumOrderModel4D, BayesOrderModel, AnchoringOrderModel

ens = HumanModelEnsemble([QuantumOrderModel4D, BayesOrderModel, AnchoringOrderModel]).update(counts, rng=rng)
for s in ens.summary():
    print('%-22s weight %.3f  BIC %.1f' % (s['model'], s['weight'], s['bic']))
p, unc = ens.predict('AB')
print('predicted answers (A first: yy, yn, ny, nn):', p.round(3))
print('uncertainty:', {k: round(v, 4) for k, v in unc.items()})

## 3. Ask or act?

In [ ]:
from qlcog.applications.robotics import ask_or_act

p_confirm = p[0] + p[1]
for error_cost in (1.5, 3.0):
    print('error cost %.1f ->' % error_cost, ask_or_act(p_confirm, unc, ask_cost=1.0, error_cost=error_cost))

## 4. Trust on the Bloch sphere

The open-system belief model treats trust as a qubit: |0⟩ (north pole) means "I trust the robot".
Successes rotate the belief towards |0⟩, failures away from it, and dephasing shrinks the vector.

In [ ]:
from qlcog.applications.robotics import TRUST_EVENTS, TRUST_MODELS
from qlcog.families.dynamics import final_yes, question_effect
from qlcog.viz import belief_trajectory, animate_bloch

model = TRUST_MODELS['open_system']
traj = belief_trajectory(model, TRUST_EVENTS, steps=10)
print('P(trust) at the end, from the sphere: %.4f; from the model: %.4f'
      % ((1 + traj.vectors[-1, 0, 2]) / 2, final_yes(model, TRUST_EVENTS, (5,))))
print('effect of also asking about trust midway: %.3f' % question_effect(model, TRUST_EVENTS, 5, 2))
animate_bloch(traj, title='Trust belief (simulated person)')